# Lab 2: Clinical data

**AI2Practice Research Lab** | about 2 hours

**The question.** Are patients sent for CT older than those sent for MRI?

By the end you will be able to clean a messy clinical table, describe it the way a paper's Table 1 does, compare groups with charts, and run and interpret a t-test and a chi-square test. You need the basics from Lab 1 (variables, DataFrames, `value_counts()`, filtering).

This lab has 4 parts: Part 1, Cleaning messy data; Part 2, Describing it: Table 1; Part 3, Charts; Part 4, Statistical tests.


#### How the tasks work

Each task is tagged with a **mode** that tells you what kind of work it is:

| Mode | What you do |
|---|---|
| **Write** | Write a short piece of code in an empty cell. Try it on your own first; a hint and a hidden solution are there if you need them. |
| **Fill in** | Some code is written for you. Replace each `____` with the missing piece. |
| **Reuse** | Import a function you wrote in an earlier lab from your own `toolkit.py`. |
| **Direct the AI** | Ask an AI assistant to write the code, paste it in, run the test, and fill in a short AI-use log. |
| **Fix** | Code "written by an AI" has one bug. Find it and fix it. |
| **Read** | Read some code and predict what it does before you run it. |

Every task ends with a **test cell**. Run it after your code. If it prints `Check passed`, move on. If it shows an `AssertionError`, the message under it tells you what to look at; change your code and run both cells again.

## Set your GitHub username

Your practice data is generated from your GitHub username, so everyone gets a slightly different dataset and the lab page can check your own numbers. Type your username between the quotes (the part after `github.com/` in your profile address), then run the cell with **Shift + Enter**.

In [ ]:
GITHUB_USERNAME = "your-username"   # <- change this to your GitHub username

## Setup

The next cell gets everything ready and also imports **SciPy**, a library for statistics. Run it once each time you open the notebook. It downloads two small helper files from the course repository if they are not here yet: `rl_data.py` makes your practice data, and `toolkit.py` is your own function library (more on that later). Lines starting with `!` are commands for the computer Colab runs on, not Python.

In [ ]:
# Setup: get the helper files (only if they are missing), then make your data
import os
COURSE = "https://raw.githubusercontent.com/UM-IHC-CA2i/CA2i-Research-Launchpad/main/lesson-notebooks"
FORK = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/CA2i-Research-Launchpad/main/lesson-notebooks"
if not os.path.exists("rl_data.py"):
    !wget -q {COURSE}/rl_data.py
if not os.path.exists("toolkit.py"):
    # Your fork first, so functions you saved in earlier labs come back; else the course copy.
    !wget -q -O toolkit.py {FORK}/toolkit.py || wget -q -O toolkit.py {COURSE}/toolkit.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rl_data
import toolkit
from toolkit import save_to_toolkit

if GITHUB_USERNAME.strip().lower() in ("", "your-username"):
    print("Reminder: put your own GitHub username in the cell above, then run both cells again.")
rl_data.save(2, GITHUB_USERNAME, "clinical.csv")   # your messy patient table
from scipy import stats
print("Ready. Data made for GitHub user:", rl_data.normalize_username(GITHUB_USERNAME))

---
## Part 1: Cleaning messy data

Real clinical data is messy. It is typed by busy people, copied between systems and merged from several sources. Before any analysis you have to **clean** it: find the problems and fix them with clear rules that you write down, so someone else could repeat your steps exactly.

Your file `clinical.csv` lists patients referred for cross-sectional imaging, with four columns: `patient_id`, `modality` (CT or MRI), `age` in years, and `sex`. Load it and look around.

In [ ]:
raw = pd.read_csv("clinical.csv")
print("Rows:", len(raw))
raw.head(10)

Three quick checks reveal most problems. `value_counts()` on a text column shows every spelling in use. `describe()` on a number column shows the smallest and largest values, which catches impossible entries. `isna().sum()` counts missing values (pandas shows a missing value as `NaN`, "not a number"). `duplicated().sum()` counts rows that are exact copies of an earlier row.

In [ ]:
print(raw["sex"].value_counts(), "\n")
print(raw["age"].describe(), "\n")
print("Missing ages:", raw["age"].isna().sum())
print("Duplicate rows:", raw.duplicated().sum())

**Mode: Read**

#### Task 1. Predict a text clean-up

Text columns have handy tools under `.str`. `.str[0]` takes the first letter of each value and `.str.upper()` makes it a capital. Predict what this prints, then write your answer as a Python list in `prediction`.

```python
pd.Series(["male", "F", "m", "Female"]).str[0].str.upper().tolist()
```

In [ ]:
prediction = ____   # for example ["X", "Y", "Z", "W"]

<details><summary>Show the answer</summary>

First letters are `m`, `F`, `m`, `F`; as capitals that is `["M", "F", "M", "F"]`. This one line turns all six spellings in your data into just M and F.

```python
prediction = ["M", "F", "M", "F"]
```

</details>

In [ ]:
assert prediction == ["M", "F", "M", "F"], "Take the first letter of each value, then make it a capital: male -> M, F -> F, m -> M, Female -> F."
print("Check passed")

### The cleaning rules

Apply these four rules, in this order. Writing rules down like this (and reporting them in your methods section) is what makes an analysis reproducible.

1. **Remove exact duplicate rows.** The same patient was entered twice.
2. **Standardize sex** to the single letters `M` and `F`.
3. **Treat impossible ages as missing.** Any age below 0 or above 120 (such as 999, a common "unknown" code) becomes `NaN`.
4. **Drop rows with a missing age**, because age is our main variable. (In a real study you would also report how many you dropped and why.)

**Mode: Fill in**

#### Task 2. Rule 1: remove duplicates

`drop_duplicates()` returns the table without exact duplicate rows. `.copy()` makes it a fresh table, so later changes do not touch `raw`. Fill in the blank, and count how many rows were removed.

In [ ]:
df = raw.____().copy()
n_removed = len(raw) - len(df)
print("Removed", n_removed, "duplicate rows;", len(df), "rows left")

<details><summary>Show a solution</summary>

```python
df = raw.drop_duplicates().copy()
n_removed = len(raw) - len(df)
print("Removed", n_removed, "duplicate rows;", len(df), "rows left")
```

</details>

In [ ]:
assert rl_data.check(2, GITHUB_USERNAME, "n_dups", n_removed), "Use raw.drop_duplicates()."
assert df.duplicated().sum() == 0
print("Check passed")

**Mode: Fill in**

#### Task 3. Rule 2: standardize sex

Use what you predicted in Task 1. Replace the blank so every value becomes `M` or `F`.

In [ ]:
df["sex"] = df["sex"].str[0].str.____()
print(df["sex"].value_counts())

<details><summary>Show a solution</summary>

```python
df["sex"] = df["sex"].str[0].str.upper()
print(df["sex"].value_counts())
```

</details>

In [ ]:
assert set(df["sex"]) == {"M", "F"}, "After this step the only values should be M and F."
print("Check passed")

**Mode: Write**

#### Task 4. Rules 3 and 4: impossible and missing ages

Set ages below 0 or above 120 to missing, then drop the rows with a missing age. Store the result in a new table called `clean`, and print how many rows it has.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* `df.loc[condition, "age"] = np.nan` changes the age only in the rows where the condition is True. Combine two conditions with `|` (or), each in brackets: `(df["age"] < 0) | (df["age"] > 120)`. Then `df.dropna(subset=["age"])` drops rows with a missing age.

In [ ]:
# Your code here

<details><summary>Show a solution</summary>

```python
df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan
clean = df.dropna(subset=["age"]).copy()
print("Rows after cleaning:", len(clean))
```

</details>

In [ ]:
assert clean["age"].isna().sum() == 0, "clean should have no missing ages."
assert clean["age"].between(0, 120).all(), "clean still contains an impossible age."
assert rl_data.check(2, GITHUB_USERNAME, "rows_clean", len(clean)), "The number of rows is off. Did you apply rules 1 to 3 before dropping missing ages?"
print("Rows after cleaning:", len(clean))
print("Check passed")

---
## Part 2: Describing it: Table 1

Almost every clinical paper starts its results with **Table 1**: who was in the study, described group by group. For a number such as age it reports a typical value and the spread:

- The **mean** is the average (add up, divide by how many). The **median** is the middle value when sorted; it is less affected by a few extreme values.
- The **standard deviation (SD)** says how far values typically sit from the mean. Roughly two thirds of patients fall within one SD of the mean when the values are bell-shaped.
- The **interquartile range (IQR)** runs from the 25th to the 75th percentile: the middle half of patients.

For a category such as sex it reports counts and percentages. `groupby()` splits the table into groups and computes something for each: `clean.groupby("modality")["age"].mean()` gives the mean age in each modality.

In [ ]:
print(clean["age"].describe().round(1), "\n")
print(clean.groupby("modality")["age"].mean().round(1))

**Mode: Write**

#### Task 5. A toolkit function: describe_by()

Write `describe_by(df, group, column)`. It should return a table with one row per value of `group` and the columns `N`, `mean`, `SD`, `median`, `min`, `max` for `column`, rounded to 1 decimal. Then run it as `table1 = describe_by(clean, "modality", "age")` and look at the result. The test saves it into your toolkit.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* `df.groupby(group)[column].agg(["count", "mean", "std", "median", "min", "max"])` computes all six at once. Then rename the columns by assigning a list to `out.columns`, and return `out.round(1)`.

In [ ]:
# Your code here


table1 = describe_by(clean, "modality", "age")
table1

<details><summary>Show a solution</summary>

```python
def describe_by(df, group, column):
    """Table 1 style summary of a number column for each group."""
    out = df.groupby(group)[column].agg(["count", "mean", "std", "median", "min", "max"])
    out.columns = ["N", "mean", "SD", "median", "min", "max"]
    return out.round(1)


table1 = describe_by(clean, "modality", "age")
table1
```

</details>

In [ ]:
assert list(table1.columns) == ["N", "mean", "SD", "median", "min", "max"], "Columns should be N, mean, SD, median, min, max (in that order)."
assert rl_data.check(2, GITHUB_USERNAME, "n_ct", table1.loc["CT", "N"]), "N for CT is off."
assert rl_data.check(2, GITHUB_USERNAME, "mean_mri", table1.loc["MRI", "mean"], tol=0.05), "The MRI mean is off."
print("Check passed")
save_to_toolkit(describe_by)

**Mode: Reuse**

#### Task 6. Sex by modality with count_by()

Table 1 also needs the sex mix in each group. Import `count_by()` from your toolkit (you wrote it in Lab 1) and use it on the CT rows and the MRI rows. Then compute the percentage of women in each group. If the import fails with "still a stub", your saved `toolkit.py` did not reach your fork. Open your Lab 1 notebook, copy your `count_by()` cell here, run it, and carry on (then save it with `save_to_toolkit(count_by)` and upload `toolkit.py` as in the Save to GitHub steps).

In [ ]:
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import count_by

ct_sex = count_by(clean[clean["modality"] == "CT"], "sex")
mri_sex = ____
pct_female_ct = round(100 * ct_sex["F"] / ct_sex.sum(), 1)
pct_female_mri = ____
print("Women: CT", pct_female_ct, "%  MRI", pct_female_mri, "%")

<details><summary>Show a solution</summary>

```python
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import count_by

ct_sex = count_by(clean[clean["modality"] == "CT"], "sex")
mri_sex = count_by(clean[clean["modality"] == "MRI"], "sex")
pct_female_ct = round(100 * ct_sex["F"] / ct_sex.sum(), 1)
pct_female_mri = round(100 * mri_sex["F"] / mri_sex.sum(), 1)
print("Women: CT", pct_female_ct, "%  MRI", pct_female_mri, "%")
```

</details>

In [ ]:
assert ct_sex.sum() + mri_sex.sum() == len(clean), "The CT and MRI counts should add up to all cleaned rows."
assert rl_data.check(2, GITHUB_USERNAME, "n_female", ct_sex["F"] + mri_sex["F"])
assert abs(pct_female_mri - 100 * mri_sex["F"] / mri_sex.sum()) < 0.06
print("Check passed")

**Mode: Fill in**

#### Task 7. Pull out the two means

`table1.loc[row, column]` reads one cell of a table. Store the mean age of each group, rounded to 1 decimal: these are two of your report-back numbers.

In [ ]:
mean_ct = table1.loc["CT", ____]
mean_mri = table1.loc[____, "mean"]
print("Mean age CT:", mean_ct, " MRI:", mean_mri, " difference:", round(mean_ct - mean_mri, 1), "years")

<details><summary>Show a solution</summary>

```python
mean_ct = table1.loc["CT", "mean"]
mean_mri = table1.loc["MRI", "mean"]
print("Mean age CT:", mean_ct, " MRI:", mean_mri, " difference:", round(mean_ct - mean_mri, 1), "years")
```

</details>

In [ ]:
assert rl_data.check(2, GITHUB_USERNAME, "mean_ct", mean_ct, tol=0.05), "mean_ct is off; read the CT row, mean column."
assert rl_data.check(2, GITHUB_USERNAME, "mean_mri", mean_mri, tol=0.05), "mean_mri is off; read the MRI row, mean column."
print("Check passed")

---
## Part 3: Charts

Summary numbers can hide the shape of the data, so always look at it too. A **histogram** splits ages into bins and shows how many patients fall in each, which shows the whole distribution. Below, the two groups are drawn on top of each other, partly see-through (`alpha=0.5`).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for modality, color in [("CT", "#0D9488"), ("MRI", "#F59E0B")]:
    ages = clean[clean["modality"] == modality]["age"]
    ax.hist(ages, bins=15, alpha=0.5, color=color, label=modality)
ax.set_title("Age of patients sent for CT and MRI")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of patients")
ax.legend()
plt.show()

**Mode: Write**

#### Task 8. A box plot

A **box plot** summarizes each group in one shape: the line in the box is the median, the box is the IQR, the whiskers reach the most extreme values that are not unusual, and dots are outliers. Draw a box plot of `age` by `modality` from `clean`, with a title and a y-axis label. Store the axes in `ax`.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* Make `fig, ax = plt.subplots()`, then `clean.boxplot(column="age", by="modality", ax=ax)`. pandas adds an automatic extra title; `plt.suptitle("")` removes it.

In [ ]:
# Your code here

<details><summary>Show a solution</summary>

```python
fig, ax = plt.subplots(figsize=(6, 4))
clean.boxplot(column="age", by="modality", ax=ax)
plt.suptitle("")   # remove the automatic extra title
ax.set_title("Age by modality")
ax.set_xlabel("Modality")
ax.set_ylabel("Age (years)")
plt.show()
```

</details>

In [ ]:
assert ax.get_title() != "", "Give the chart a title with ax.set_title()."
assert ax.get_ylabel() != "", "Label the y axis with ax.set_ylabel()."
print("Check passed")

**Mode: Read**

#### Task 9. What does this code show?

Read this code and decide what it prints. Set `prediction` to the letter.

```python
older = clean[clean["age"] >= 80]
older.groupby("modality").size()
```

- A: the mean age in each modality
- B: how many patients aged 80 or over were sent for each modality
- C: the 80th percentile of age in each modality

In [ ]:
prediction = "____"

<details><summary>Show the answer</summary>

B. The first line filters to patients aged 80 or over, and `groupby("modality").size()` counts the rows in each modality.

```python
prediction = "B"
```

</details>

In [ ]:
assert prediction.strip().upper() == "B", "The first line keeps patients 80 or older; groupby(...).size() counts rows in each group."
print(clean[clean["age"] >= 80].groupby("modality").size())
print("Check passed")

---
## Part 4: Statistical tests

Your CT patients are older on average than your MRI patients. But any two samples differ a little just by chance. A **statistical test** asks how surprising the difference would be if, in truth, there were no difference at all. That "no difference" assumption is the **null hypothesis**.

The test returns a **p-value**: the probability of seeing a difference at least this large if the null hypothesis were true. A small p-value (by convention below 0.05) means the difference would be surprising under "no difference", so we call it **statistically significant**. A p-value is not the probability that the result is true, and it says nothing about whether the difference matters clinically. Always report the size of the difference (here, in years) next to it.

The **t-test** compares the means of a number between two groups. The **chi-square test** compares proportions of a category (such as sex) between groups.

**Mode: Fill in**

#### Task 10. A t-test for age

`stats.ttest_ind(a, b)` runs a t-test comparing two lists of numbers and returns the t statistic and the p-value. Fill in the blanks.

In [ ]:
ct_ages = clean[clean["modality"] == "CT"]["age"]
mri_ages = clean[clean["modality"] == ____]["age"]
t_stat, p_value = stats.ttest_ind(ct_ages, ____)
print(f"t = {t_stat:.2f}, p = {p_value:.4f}")

<details><summary>Show a solution</summary>

```python
ct_ages = clean[clean["modality"] == "CT"]["age"]
mri_ages = clean[clean["modality"] == "MRI"]["age"]
t_stat, p_value = stats.ttest_ind(ct_ages, mri_ages)
print(f"t = {t_stat:.2f}, p = {p_value:.4f}")
```

</details>

In [ ]:
assert rl_data.check(2, GITHUB_USERNAME, "n_mri", len(mri_ages)), "mri_ages should hold the ages of the MRI patients."
assert 0 <= p_value <= 1
assert abs(t_stat - stats.ttest_ind(ct_ages, mri_ages)[0]) < 1e-9, "Compare ct_ages with mri_ages."
print("Check passed")

**Mode: Write**

#### Task 11. A chi-square test for sex

Is the sex mix different between CT and MRI? Make a **contingency table** (counts for each combination of modality and sex) with `pd.crosstab()`, store it in `tab`, then run `stats.chi2_contingency(tab)` and store the p-value in `p_chi`.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* `pd.crosstab(clean["modality"], clean["sex"])` builds the table. `stats.chi2_contingency(tab)` returns four things; the second is the p-value: `chi2, p_chi, dof, expected_counts = stats.chi2_contingency(tab)`.

In [ ]:
# Your code here

<details><summary>Show a solution</summary>

```python
tab = pd.crosstab(clean["modality"], clean["sex"])
print(tab)
chi2, p_chi, dof, expected_counts = stats.chi2_contingency(tab)
print(f"chi-square = {chi2:.2f}, p = {p_chi:.3f}")
```

</details>

In [ ]:
assert tab.shape == (2, 2), "tab should be a 2 x 2 table: modality by sex."
assert tab.values.sum() == len(clean)
assert abs(p_chi - stats.chi2_contingency(tab)[1]) < 1e-12
print(f"p = {p_chi:.3f}")
print("Check passed")

### Putting it into words

A results sentence for this lab could read: "Patients referred for CT were older than those referred for MRI (mean 61 vs 54 years; t-test p < 0.001). The proportion of women did not differ (chi-square p = 0.4)." Use your own numbers. Notice what it does not claim: it does not say why CT patients are older (perhaps more emergency and cancer staging scans), and with made-up data it says nothing about real hospitals.

---
## Report back

You can now answer the question: are patients sent for CT older than those sent for MRI? Run the cell below. It prints the numbers to enter on the Lab 2 page on the AI2Practice site, together with your GitHub username (use the same one there). If the page says a number is off, it gives you a hint first.

In [ ]:
REPORT = {
    "rows_clean": len(clean),
    "mean_ct": round(float(mean_ct), 1),
    "mean_mri": round(float(mean_mri), 1),
}
print("Report back for Lab 2")
print("GitHub username:", rl_data.normalize_username(GITHUB_USERNAME))
LABELS = {
    "rows_clean": "Rows after cleaning",
    "mean_ct": "Mean age, CT (years)",
    "mean_mri": "Mean age, MRI (years)",
}
for key, value in REPORT.items():
    print(f"  {LABELS[key]}: {value}")

---
## Save to GitHub

**1. Save the notebook.** In Colab, choose **File > Save a copy in GitHub**. The first time, Colab asks you to sign in to GitHub and allow access. Then set:

- Repository: `<your-username>/CA2i-Research-Launchpad` (your fork)
- Branch: `main`
- File path: `lesson-notebooks/Lab_2_Clinical_Data.ipynb`
- Commit message: `Lab 2 finished`

Leave "Include a link to Colab" ticked and click **OK**. A **commit** is a saved snapshot of your files with a short message, so you can always see what changed and go back.

**2. Save your toolkit.** Colab forgets files like `toolkit.py` when the session ends, so keep your copy on GitHub too:

1. In the left sidebar of Colab, click the folder icon (Files). Find `toolkit.py`, click the three dots next to it, and choose **Download**.
2. On GitHub, open your fork (`github.com/<your-username>/CA2i-Research-Launchpad`) and click into the `lesson-notebooks` folder.
3. Click **Add file > Upload files**, drop in `toolkit.py`, write the message `Lab 2: add describe_by()`, and click **Commit changes**. It replaces the starter copy.

Next time, the setup cell downloads `toolkit.py` from your fork, so `describe_by()` is there waiting for you.

That's it for Lab 2. Go back to the lab page, enter your report-back numbers and write one or two sentences on what you found.